# nested_data_structures
Nested data structures are where real Python work starts to look like real system data: JSON payloads, configs, grouped metrics, catalog trees, and batched records. The challenge is not syntax; it is safe traversal, mutation, and shape reasoning.

## 1. Problem First
Why does this matter in real systems?
- API responses are usually dictionaries containing lists of dictionaries.
- Config files often have deep nested mappings and optional branches.
- Bugs come from wrong assumptions about shape and shared nested mutation.

In [ ]:
payload = {
    "service": "api",
    "instances": [
        {"id": 1, "healthy": True},
        {"id": 2, "healthy": False}
    ]
}
print(payload["instances"][1]["healthy"])

## 2. Minimal Concept
Core idea:
- Lists can hold dicts
- Dicts can hold lists
- Structures can be nested to arbitrary depth
- Access combines keys and indexes step by step

## 3. Mental Model
How Python actually behaves
Nested structures are layers of references. A shallow copy only duplicates the outer container; inner containers remain shared. That means one innocent-looking update deep inside a nested structure can affect another part of the program unexpectedly.

In [ ]:
config = {"services": [{"name": "api", "ports": [80, 443]}]}
copy_config = config.copy()
copy_config["services"][0]["ports"].append(8080)
print(config)
print(copy_config)

## 4. Internal Mechanics
Nested access works one layer at a time. Each `[]` operation returns another object reference. The danger is that every inner mutation acts on the underlying referenced container, not on an imagined isolated snapshot.

In [ ]:
import dis

def second_instance_status(payload):
    return payload["instances"][1]["healthy"]

dis.dis(second_instance_status)
sample = {"instances": [{"healthy": True}, {"healthy": False}]}
print(second_instance_status(sample))

## 5. Edge Cases
Where it breaks:
- A missing intermediate key causes failure before the deeper access even matters.
- Empty lists make index-based access fragile.
- Shallow copies create aliasing bugs at inner levels.
- Overly deep structures become hard to validate and reason about.

In [ ]:
payload = {"instances": []}
try:
    print(payload["instances"][0])
except IndexError as exc:
    print(type(exc).__name__)

try:
    print(payload["meta"]["owner"])
except KeyError as exc:
    print(type(exc).__name__)

## 6. Performance Thinking
- Single access steps are usually O(1), but long chains increase failure risk and readability cost.
- Deep copies can be expensive in both time and memory.
- Flattening or indexing parts of nested structures can improve repeated access patterns.

## 7. Real Use Case
A service discovery payload may contain lists of instances, each with nested metadata and health information.

In [ ]:
discovery = {
    "clusters": [
        {"name": "api", "instances": [{"id": "a1", "healthy": True}]},
        {"name": "worker", "instances": [{"id": "w1", "healthy": False}]}
    ]
}
for cluster in discovery["clusters"]:
    for instance in cluster["instances"]:
        print(cluster["name"], instance["id"], instance["healthy"])

## 8. Anti-Pattern
What beginners do wrong:
- Hard-code long access chains everywhere instead of naming intermediate values.
- Assume the shape is stable without validation.
- Mutate deep nested values without understanding shared references.

In [ ]:
payload = {"data": {"users": [{"name": "a"}]}}
print(payload["data"]["users"][0]["name"])
user = payload["data"]["users"][0]
print(user["name"])

## 9. Interview Signals
Questions FAANG asks:
- What is the risk of shallow copying nested structures?
- How would you safely traverse optional nested fields?
- When should you flatten or normalize nested data?
- How do you validate a nested payload shape?

## 10. Exercise (Non-trivial)
Design a normalizer for an API payload containing nested lists and dictionaries. It should safely handle missing keys, empty arrays, and partially malformed records while preserving enough context to report exactly where a bad shape was found.

In [ ]:
def collect_healthy_ids(payload):
    result = []
    for cluster in payload.get("clusters", []):
        for instance in cluster.get("instances", []):
            if instance.get("healthy"):
                result.append(instance["id"])
    return result

# Task:
# 1. Add error reporting for malformed cluster shapes.
# 2. Explain where shallow copy would still be dangerous.
# 3. Decide whether normalization should flatten this structure.